In [3]:
import pandas as pd
df=pd.read_csv('../data/transactions_clean.csv')
df.head()

,transaction_id,user_id,date,description,amount,transaction_type,category,merchant
0,1,1,2025-01-01,LPG cylinder booking,896.21,expense,Bills,LPG cylinder booking
1,2,1,2025-01-01,IRCTC train ticket,1194.82,expense,Transport,IRCTC train ticket
2,3,1,2025-01-03,Salary credited,31445.84,income,Income,Employer/Institute
3,4,1,2025-01-04,McDonalds,257.03,expense,Food,McDonalds
4,5,1,2025-01-05,Ola cab,338.20,expense,Transport,Ola cab


In [4]:
print(df.shape)
print(df['category'].value_counts())

(1346, 8)
category
Food             300
Bills            205
Shopping         197
Transport        187
Education        108
Entertainment    102
Income            93
Health            93
Other             61
Name: count, dtype: int64


In [6]:
expenses = df[df['transaction_type']=='expense']
print(expenses.shape)
print(expenses['category'].value_counts())


(1253, 8)
category
Food             300
Bills            205
Shopping         197
Transport        187
Education        108
Entertainment    102
Health            93
Other             61
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

X = expenses['description']
y = expenses['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(1002,) (251,)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)

(1002, 149)


In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print("Training done!")

Training done!


In [10]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
               precision    recall  f1-score   support

        Bills       1.00      1.00      1.00        33
    Education       1.00      1.00      1.00        24
Entertainment       1.00      1.00      1.00        23
         Food       1.00      1.00      1.00        57
       Health       1.00      1.00      1.00        16
        Other       1.00      1.00      1.00        16
     Shopping       1.00      1.00      1.00        40
    Transport       1.00      1.00      1.00        42

     accuracy                           1.00       251
    macro avg       1.00      1.00      1.00       251
 weighted avg       1.00      1.00      1.00       251



In [11]:
new_transactions = [
    "Dominos garlic bread order",       # Food - but different phrasing than training
    "Ola auto to college",               # Transport - new phrasing
    "New cafe near hostel, filter coffee", # tricky - unfamiliar merchant
    "Paid electrician for wiring work",  # ambiguous - could be Bills or Other
    "Spotify family plan renewal",       # Entertainment - unseen exact phrase
    "Bought a birthday gift for friend", # Other - no strong keyword
]

new_vec = vectorizer.transform(new_transactions)
predictions = model.predict(new_vec)

for text, pred in zip(new_transactions, predictions):
    print(f"{text}  →  {pred}")

Dominos garlic bread order  →  Food
Ola auto to college  →  Transport
New cafe near hostel, filter coffee  →  Food
Paid electrician for wiring work  →  Food
Spotify family plan renewal  →  Entertainment
Bought a birthday gift for friend  →  Other


In [12]:
extra_descriptions = [
    "Paid electrician for wiring work",
    "Plumber fixed the bathroom leak",
    "AC repair and gas refill",
    "Carpenter fixed the cupboard",
    "Laptop repair service",
    "Cobbler shoe repair",
]
extra_categories = [
    "Other",
    "Other",
    "Other",
    "Other",
    "Other",
    "Other",
]

import pandas as pd

X_train_extended = pd.concat([X_train, pd.Series(extra_descriptions)], ignore_index=True)
y_train_extended = pd.concat([y_train, pd.Series(extra_categories)], ignore_index=True)

print(X_train_extended.shape, y_train_extended.shape)

(1008,) (1008,)


In [13]:
vectorizer2 = TfidfVectorizer()
X_train_vec2 = vectorizer2.fit_transform(X_train_extended)
X_test_vec2 = vectorizer2.transform(X_test)

model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train_vec2, y_train_extended)

print("Retrained!")

Retrained!


In [14]:
test_case = ["Paid electrician for wiring work"]
test_vec = vectorizer2.transform(test_case)
print(model2.predict(test_vec))

['Other']


In [15]:
y_pred2 = model2.predict(X_test_vec2)
print("Accuracy:", accuracy_score(y_test, y_pred2))

Accuracy: 1.0


In [16]:
import joblib

joblib.dump(model2, "../ml/expense_classifier.pkl")
joblib.dump(vectorizer2, "../ml/tfidf_vectorizer.pkl")

print("Saved!")

Saved!
